In [1]:
# Setup: stesso ambiente e stessa struttura del progetto di baselines.ipynb
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    GITHUB_USER = "SinghSahiljit44"
    REPO_NAME = "Weather_forecasting"
    REPO_URL = f"https://github.com/{GITHUB_USER}/{REPO_NAME}.git"

    if not os.path.exists(REPO_NAME):
        !git clone {REPO_URL}
    os.chdir(REPO_NAME)
    PROJECT_ROOT = Path.cwd()
else:
    PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    os.chdir(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print(f"Ambiente: {'Google Colab' if IN_COLAB else 'Locale'}")
print(f"Project root: {PROJECT_ROOT}")

Ambiente: Locale
Project root: c:\Users\sahil\GitHub\Weather_forecasting


In [2]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from src.data.preprocessing import load_daily
from src.utils.split import Split

# Load daily data
data = load_daily().sort_index().asfreq("D")

# Target to forecast
target = "t2m_mean"

# Calendar features
doy = data.index.dayofyear
data["doy_sin"] = np.sin(2 * np.pi * doy / 365.25)
data["doy_cos"] = np.cos(2 * np.pi * doy / 365.25)

# Historical temperature features
for k in range(1, 4):
    data[f"t2m_lag{k}"] = data[target].shift(k)

data["t2m_roll7"] = data[target].rolling(7).mean()
data["t2m_roll30"] = data[target].rolling(30).mean()

# Input features
features = [
    "t2m_mean", "t2m_min", "t2m_max",
    "tp_mm", "e_mm", "pev_mm",
    "skt_mean", "stl1_mean", "stl2_mean",
    "stl3_mean", "stl4_mean",
    "swvl1_mean", "swvl2_mean", "swvl3_mean", "swvl4_mean",
    "doy_sin", "doy_cos",
    "t2m_lag1", "t2m_lag2", "t2m_lag3",
    "t2m_roll7", "t2m_roll30"
]

print(f"Date range: {data.index.min()} -> {data.index.max()}")
print(f"Number of daily rows: {len(data):,}")
print(f"Number of features: {len(features)}")
print(f"Target: {target}")

display(data[features + [target]].head())
print("\nMissing values:")
display(data[features + [target]].isna().sum())


ImportError: DLL load failed while importing _heap: Un criterio di controllo dell'applicazione ha bloccato il file.

In [ ]:
display(data.head(30))

In [4]:
# Window configuration
INPUT_DAYS = 30
HORIZON = 30

# Convert features and target to NumPy arrays
X_raw = data[features].to_numpy(dtype=float)
y_raw = data[target].to_numpy(dtype=float)
dates = data.index

X_windows = []
Y_windows = []
forecast_start_dates = []

# Build input/target windows
for i in range(INPUT_DAYS, len(data) - HORIZON + 1):

    # Historical observations: 30 days before the forecast starts
    X_window = X_raw[i - INPUT_DAYS:i]

    # Future target: temperature for the following 30 days
    Y_window = y_raw[i:i + HORIZON]

    # Skip windows containing missing or infinite values
    if not np.isfinite(X_window).all():
        continue

    if not np.isfinite(Y_window).all():
        continue

    X_windows.append(X_window)
    Y_windows.append(Y_window)
    forecast_start_dates.append(dates[i])

# Convert lists to arrays
X_windows = np.asarray(X_windows, dtype=np.float32)
Y_windows = np.asarray(Y_windows, dtype=np.float32)
forecast_start_dates = pd.DatetimeIndex(forecast_start_dates)

print("Input shape:", X_windows.shape)
print("Target shape:", Y_windows.shape)
print("First forecast starts:", forecast_start_dates[0])
print("Last forecast starts:", forecast_start_dates[-1])

# Inspect the first example
print("\nFirst input window:")
display(data.loc[
    forecast_start_dates[0] - pd.Timedelta(days=INPUT_DAYS):
    forecast_start_dates[0] - pd.Timedelta(days=1),
    features
])

print("\nFirst target window:")
display(data.loc[
    forecast_start_dates[0]:
    forecast_start_dates[0] + pd.Timedelta(days=HORIZON - 1),
    [target]
])


Input shape: (13060, 30, 22)
Target shape: (13060, 30)
First forecast starts: 1990-03-01 00:00:00
Last forecast starts: 2025-12-01 00:00:00

First input window:


,t2m_mean,t2m_min,t2m_max,tp_mm,e_mm,pev_mm,skt_mean,stl1_mean,stl2_mean,stl3_mean,...,swvl2_mean,swvl3_mean,swvl4_mean,doy_sin,doy_cos,t2m_lag1,t2m_lag2,t2m_lag3,t2m_roll7,t2m_roll30
date,,,,,,,,,,,,,,,,,,,,,
1990-01-30,11.465077,8.107202,16.168604,0.303477,-0.705227,-1.374885,9.636458,9.718357,9.485895,9.332198,...,0.391345,0.344290,0.314011,0.493468,0.869764,11.033080,10.783243,11.238336,10.368148,6.778642
1990-01-31,10.446909,6.570947,13.612939,0.626183,-0.459140,-0.854790,9.297665,9.633808,9.738062,9.520914,...,0.390482,0.344330,0.314011,0.508356,0.861147,11.465077,11.033080,10.783243,10.830600,6.908786
1990-02-01,10.364997,5.475488,16.091821,0.001853,-0.690470,-1.524098,8.243895,8.793639,9.047378,9.640248,...,0.389559,0.344343,0.314011,0.523094,0.852275,10.446909,11.465077,11.033080,11.079753,6.996823
1990-02-02,10.450805,5.518701,16.338647,0.000829,-0.697567,-1.510449,8.855717,9.259103,9.251159,9.722208,...,0.388034,0.344276,0.314011,0.537677,0.843151,10.364997,10.446909,11.465077,10.826064,7.111003
1990-02-03,9.655257,4.966211,14.935693,0.092471,-0.459313,-1.114450,8.424460,8.989821,9.321308,9.813603,...,0.386630,0.344227,0.314011,0.552101,0.833777,10.450805,10.364997,10.446909,10.599910,7.242234
1990-02-04,8.930828,4.045313,14.447412,0.115860,-0.447689,-1.322411,7.913797,8.440083,8.832381,9.859908,...,0.385478,0.344207,0.314011,0.566362,0.824157,9.655257,10.450805,10.364997,10.335279,7.374268
1990-02-05,8.405450,2.359033,13.331506,0.009495,-0.773942,-1.936629,8.216119,9.015176,9.302779,9.902231,...,0.384089,0.344147,0.314012,0.580455,0.814292,8.930828,9.655257,10.450805,9.959903,7.495459
1990-02-06,6.403894,0.674951,14.732874,0.001291,-0.871954,-2.234668,6.151890,7.431075,8.371108,9.905766,...,0.382229,0.344014,0.314011,0.594376,0.804187,8.405450,8.930828,9.655257,9.236877,7.519251
1990-02-07,6.652767,0.522119,13.577295,0.010967,-0.815767,-1.886167,6.366907,7.402322,8.060663,9.840707,...,0.380245,0.343856,0.314011,0.608121,0.793844,6.403894,8.405450,8.930828,8.694857,7.558518



First target window:


,t2m_mean
date,
1990-03-01,14.001210
1990-03-02,13.027317
1990-03-03,9.681177
1990-03-04,7.412017
1990-03-05,7.973235
1990-03-06,8.480488
1990-03-07,9.811151
1990-03-08,11.401371
1990-03-09,11.070622


In [5]:
# Split windows chronologically, keeping each 30-day target
# entirely inside its assigned period.

target_end_dates = (
    forecast_start_dates + pd.Timedelta(days=HORIZON - 1)
)

train_mask = (
    (forecast_start_dates.year <= 2017)
    & (target_end_dates.year <= 2017)
)

val_mask = (
    (forecast_start_dates.year >= 2018)
    & (forecast_start_dates.year <= 2021)
    & (target_end_dates.year <= 2021)
)

test_mask = forecast_start_dates.year >= 2022

# Apply masks
X_train_seq, Y_train = X_windows[train_mask], Y_windows[train_mask]
X_val_seq, Y_val = X_windows[val_mask], Y_windows[val_mask]
X_test_seq, Y_test = X_windows[test_mask], Y_windows[test_mask]

# Corresponding forecast start dates
dates_train = forecast_start_dates[train_mask]
dates_val = forecast_start_dates[val_mask]
dates_test = forecast_start_dates[test_mask]

# Display split information
for name, X_part, Y_part, dates_part in [
    ("Train", X_train_seq, Y_train, dates_train),
    ("Validation", X_val_seq, Y_val, dates_val),
    ("Test", X_test_seq, Y_test, dates_test),
]:
    print(f"{name}:")
    print(f"  Samples: {len(X_part):,}")
    print(f"  Input shape: {X_part.shape}")
    print(f"  Target shape: {Y_part.shape}")
    if len(dates_part):
        print(f"  Forecast start: {dates_part.min().date()}")
        print(f"  Forecast end:   {dates_part.max().date()}")
    print()


Train:
  Samples: 10,139
  Input shape: (10139, 30, 22)
  Target shape: (10139, 30)
  Forecast start: 1990-03-01
  Forecast end:   2017-12-02

Validation:
  Samples: 1,432
  Input shape: (1432, 30, 22)
  Target shape: (1432, 30)
  Forecast start: 2018-01-01
  Forecast end:   2021-12-02

Test:
  Samples: 1,431
  Input shape: (1431, 30, 22)
  Target shape: (1431, 30)
  Forecast start: 2022-01-01
  Forecast end:   2025-12-01



In [6]:
# Flatten the 30-day input windows for scikit-learn models

X_train = X_train_seq.reshape(X_train_seq.shape[0], -1)
X_val = X_val_seq.reshape(X_val_seq.shape[0], -1)
X_test = X_test_seq.reshape(X_test_seq.shape[0], -1)

print("X_train:", X_train.shape)
print("Y_train:", Y_train.shape)

print("X_val:", X_val.shape)
print("Y_val:", Y_val.shape)

print("X_test:", X_test.shape)
print("Y_test:", Y_test.shape)


X_train: (10139, 660)
Y_train: (10139, 30)
X_val: (1432, 660)
Y_val: (1432, 30)
X_test: (1431, 660)
Y_test: (1431, 30)


In [7]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Create and train the model
lr_model = LinearRegression()
lr_model.fit(X_train, Y_train)

# Predict the 30 future temperatures on the validation set
Y_val_pred = lr_model.predict(X_val)

print("Predictions shape:", Y_val_pred.shape)

Predictions shape: (1432, 30)


In [8]:
mae = mean_absolute_error(Y_val, Y_val_pred)
rmse = np.sqrt(mean_squared_error(Y_val, Y_val_pred))
r2 = r2_score(Y_val, Y_val_pred, multioutput="uniform_average")

print(f"Validation MAE:  {mae:.3f} °C")
print(f"Validation RMSE: {rmse:.3f} °C")
print(f"Validation R²:   {r2:.3f}")

Validation MAE:  2.173 °C
Validation RMSE: 2.734 °C
Validation R²:   0.875


In [9]:
mae_by_horizon = np.mean(np.abs(Y_val - Y_val_pred), axis=0)
rmse_by_horizon = np.sqrt(np.mean((Y_val - Y_val_pred) ** 2, axis=0))

for h in [1, 7, 15, 30]:
    print(
        f"Giorno +{h:02d} | "
        f"MAE: {mae_by_horizon[h - 1]:.3f} °C | "
        f"RMSE: {rmse_by_horizon[h - 1]:.3f} °C"
    )

Giorno +01 | MAE: 1.097 °C | RMSE: 1.423 °C
Giorno +07 | MAE: 2.207 °C | RMSE: 2.777 °C
Giorno +15 | MAE: 2.269 °C | RMSE: 2.825 °C
Giorno +30 | MAE: 2.279 °C | RMSE: 2.833 °C


In [10]:
# Last observed temperature in each 30-day input window
last_temperature = X_val_seq[:, -1, features.index(target)]

# Predict the same temperature for all 30 future days
Y_val_persistence = np.repeat(
    last_temperature[:, np.newaxis],
    HORIZON,
    axis=1
)

# Overall validation metrics
persistence_mae = mean_absolute_error(Y_val, Y_val_persistence)
persistence_rmse = np.sqrt(
    mean_squared_error(Y_val, Y_val_persistence)
)
persistence_r2 = r2_score(
    Y_val, Y_val_persistence, multioutput="uniform_average"
)

print(f"Persistence MAE:  {persistence_mae:.3f} °C")
print(f"Persistence RMSE: {persistence_rmse:.3f} °C")
print(f"Persistence R²:   {persistence_r2:.3f}")

# Compare errors at selected forecast horizons
print("\nComparison by forecast horizon:")

for h in [1, 7, 15, 30]:
    lr_mae = mean_absolute_error(Y_val[:, h - 1], Y_val_pred[:, h - 1])
    persistence_h_mae = mean_absolute_error(
        Y_val[:, h - 1], Y_val_persistence[:, h - 1]
    )

    print(
        f"Day +{h:02d} | "
        f"Linear Regression MAE: {lr_mae:.3f} °C | "
        f"Persistence MAE: {persistence_h_mae:.3f} °C"
    )

Persistence MAE:  3.283 °C
Persistence RMSE: 4.207 °C
Persistence R²:   0.704

Comparison by forecast horizon:
Day +01 | Linear Regression MAE: 1.097 °C | Persistence MAE: 1.253 °C
Day +07 | Linear Regression MAE: 2.207 °C | Persistence MAE: 2.736 °C
Day +15 | Linear Regression MAE: 2.269 °C | Persistence MAE: 3.309 °C
Day +30 | Linear Regression MAE: 2.279 °C | Persistence MAE: 4.350 °C


In [11]:
# Use only historical observations available in the training period
train_history = data.loc[data.index.year <= 2017, target]

# Average temperature for each month-day, computed on training data only
climatology = train_history.groupby(
    train_history.index.strftime("%m-%d")
).mean()

# Build climatological predictions for the validation target dates
Y_val_climatology = np.empty_like(Y_val)

for i, start_date in enumerate(dates_val):
    future_dates = pd.date_range(
        start=start_date,
        periods=HORIZON,
        freq="D"
    )

    month_days = future_dates.strftime("%m-%d")
    predictions = climatology.reindex(month_days).to_numpy()

    # Handle February 29 if it was not present in the training climatology
    for j, date in enumerate(future_dates):
        if np.isnan(predictions[j]):
            if date.strftime("%m-%d") == "02-29":
                predictions[j] = (
                    climatology["02-28"] + climatology["03-01"]
                ) / 2
            else:
                raise ValueError(
                    f"Missing climatology for {date.strftime('%m-%d')}"
                )

    Y_val_climatology[i] = predictions

print("Climatology predictions shape:", Y_val_climatology.shape)

print(
    f"Climatology MAE:  "
    f"{mean_absolute_error(Y_val, Y_val_climatology):.3f} °C"
)
print(
    f"Climatology RMSE: "
    f"{np.sqrt(mean_squared_error(Y_val, Y_val_climatology)):.3f} °C"
)
print(
    f"Climatology R²:   "
    f"{r2_score(Y_val, Y_val_climatology):.3f}"
)

Climatology predictions shape: (1432, 30)
Climatology MAE:  2.183 °C
Climatology RMSE: 2.754 °C
Climatology R²:   0.873


In [12]:
models_val = {
    "Linear Regression": Y_val_pred,
    "Persistence": Y_val_persistence,
    "Climatology": Y_val_climatology,
}

horizons = [1, 7, 15, 30]

for h in horizons:
    print(f"\nForecast horizon: +{h} days")

    for name, predictions in models_val.items():
        mae = mean_absolute_error(
            Y_val[:, h - 1],
            predictions[:, h - 1]
        )
        rmse = np.sqrt(mean_squared_error(
            Y_val[:, h - 1],
            predictions[:, h - 1]
        ))

        print(f"{name:20s} MAE={mae:.3f} °C | RMSE={rmse:.3f} °C")


Forecast horizon: +1 days
Linear Regression    MAE=1.097 °C | RMSE=1.423 °C
Persistence          MAE=1.253 °C | RMSE=1.627 °C
Climatology          MAE=2.193 °C | RMSE=2.763 °C

Forecast horizon: +7 days
Linear Regression    MAE=2.207 °C | RMSE=2.777 °C
Persistence          MAE=2.736 °C | RMSE=3.482 °C
Climatology          MAE=2.185 °C | RMSE=2.757 °C

Forecast horizon: +15 days
Linear Regression    MAE=2.269 °C | RMSE=2.825 °C
Persistence          MAE=3.309 °C | RMSE=4.133 °C
Climatology          MAE=2.182 °C | RMSE=2.753 °C

Forecast horizon: +30 days
Linear Regression    MAE=2.279 °C | RMSE=2.833 °C
Persistence          MAE=4.350 °C | RMSE=5.372 °C
Climatology          MAE=2.192 °C | RMSE=2.762 °C


In [13]:
from sklearn.ensemble import RandomForestRegressor

rf_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=15,
    min_samples_leaf=3,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, Y_train)

Y_val_rf_pred = rf_model.predict(X_val)

print(f"Random Forest MAE:  {mean_absolute_error(Y_val, Y_val_rf_pred):.3f} °C")
print(f"Random Forest RMSE: {np.sqrt(mean_squared_error(Y_val, Y_val_rf_pred)):.3f} °C")
print(f"Random Forest R²:   {r2_score(Y_val, Y_val_rf_pred):.3f}")

Random Forest MAE:  2.322 °C
Random Forest RMSE: 2.951 °C
Random Forest R²:   0.854


In [14]:
Y_train_pred = lr_model.predict(X_train)

train_mae = mean_absolute_error(Y_train, Y_train_pred)
train_rmse = np.sqrt(mean_squared_error(Y_train, Y_train_pred))
train_r2 = r2_score(Y_train, Y_train_pred)

print(f"Training MAE:  {train_mae:.3f} °C")
print(f"Training RMSE: {train_rmse:.3f} °C")
print(f"Training R²:   {train_r2:.3f}")


Training MAE:  2.066 °C
Training RMSE: 2.599 °C
Training R²:   0.885


In [15]:

horizons = [1, 7, 15, 30]

rows = []

for h in horizons:
    idx = h - 1

    rows.append({
        "Horizon (days)": h,
        "Train MAE (°C)": mean_absolute_error(
            Y_train[:, idx],
            lr_model.predict(X_train)[:, idx]
        ),
        "Validation MAE (°C)": mean_absolute_error(
            Y_val[:, idx],
            lr_model.predict(X_val)[:, idx]
        ),
    })

horizon_comparison = pd.DataFrame(rows)
display(horizon_comparison.round(3))


,Horizon (days),Train MAE (°C),Validation MAE (°C)
0,1,1.028,1.097
1,7,2.101,2.207
2,15,2.143,2.269
3,30,2.153,2.279


In [16]:

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

INPUT_DAYS = 14
HORIZON = 30

X_raw = data[features].to_numpy(dtype=float)
y_raw = data[target].to_numpy(dtype=float)
dates = data.index

X_windows, Y_windows, start_dates = [], [], []

for i in range(INPUT_DAYS, len(data) - HORIZON + 1):
    X_window = X_raw[i - INPUT_DAYS:i]
    Y_window = y_raw[i:i + HORIZON]

    if not np.isfinite(X_window).all():
        continue
    if not np.isfinite(Y_window).all():
        continue

    X_windows.append(X_window)
    Y_windows.append(Y_window)
    start_dates.append(dates[i])

X_windows = np.asarray(X_windows, dtype=np.float32)
Y_windows = np.asarray(Y_windows, dtype=np.float32)
start_dates = pd.DatetimeIndex(start_dates)

end_dates = start_dates + pd.Timedelta(days=HORIZON - 1)

train_mask = (start_dates.year <= 2017) & (end_dates.year <= 2017)
val_mask = (
    (start_dates.year >= 2018)
    & (start_dates.year <= 2021)
    & (end_dates.year <= 2021)
)

X_flat = X_windows.reshape(len(X_windows), -1)

X_train14, Y_train14 = X_flat[train_mask], Y_windows[train_mask]
X_val14, Y_val14 = X_flat[val_mask], Y_windows[val_mask]

model14 = LinearRegression()
model14.fit(X_train14, Y_train14)

pred_train14 = model14.predict(X_train14)
pred_val14 = model14.predict(X_val14)

# Metriche per training e validation
rows = []

for name, actual, predicted in [
    ("Training", Y_train14, pred_train14),
    ("Validation", Y_val14, pred_val14),
]:
    rows.append({
        "Dataset": name,
        "MAE (°C)": mean_absolute_error(actual, predicted),
        "RMSE (°C)": np.sqrt(mean_squared_error(actual, predicted)),
        "R²": r2_score(actual, predicted),
    })

print("Input: 14 giorni | Forecast: 30 giorni")
display(pd.DataFrame(rows).set_index("Dataset").round(3))

# MAE ai diversi orizzonti
rows = []

for h in [1, 7, 15, 30]:
    idx = h - 1
    rows.append({
        "Horizon (days)": h,
        "Train MAE (°C)": mean_absolute_error(
            Y_train14[:, idx], pred_train14[:, idx]
        ),
        "Validation MAE (°C)": mean_absolute_error(
            Y_val14[:, idx], pred_val14[:, idx]
        ),
    })

display(pd.DataFrame(rows).round(3))


Input: 14 giorni | Forecast: 30 giorni


,MAE (°C),RMSE (°C),R²
Dataset,,,
Training,2.096,2.633,0.882
Validation,2.173,2.734,0.875


,Horizon (days),Train MAE (°C),Validation MAE (°C)
0,1,1.043,1.087
1,7,2.136,2.204
2,15,2.165,2.280
3,30,2.191,2.259


In [18]:

INPUT_DAYS = 1
HORIZON = 30

X_raw = data[features].to_numpy(dtype=float)
y_raw = data[target].to_numpy(dtype=float)
dates = data.index

X_windows, Y_windows, start_dates = [], [], []

for i in range(INPUT_DAYS, len(data) - HORIZON + 1):
    X_window = X_raw[i - INPUT_DAYS:i]
    Y_window = y_raw[i:i + HORIZON]

    if not np.isfinite(X_window).all():
        continue
    if not np.isfinite(Y_window).all():
        continue

    X_windows.append(X_window)
    Y_windows.append(Y_window)
    start_dates.append(dates[i])

X_windows = np.asarray(X_windows, dtype=np.float32)
Y_windows = np.asarray(Y_windows, dtype=np.float32)
start_dates = pd.DatetimeIndex(start_dates)

end_dates = start_dates + pd.Timedelta(days=HORIZON - 1)

train_mask = (start_dates.year <= 2017) & (end_dates.year <= 2017)
val_mask = (
    (start_dates.year >= 2018)
    & (start_dates.year <= 2021)
    & (end_dates.year <= 2021)
)

X_flat = X_windows.reshape(len(X_windows), -1)

X_train7, Y_train7 = X_flat[train_mask], Y_windows[train_mask]
X_val7, Y_val7 = X_flat[val_mask], Y_windows[val_mask]

model7 = LinearRegression()
model7.fit(X_train7, Y_train7)

pred_train7 = model7.predict(X_train7)
pred_val7 = model7.predict(X_val7)

# Metriche complessive
rows = []

for name, actual, predicted in [
    ("Training", Y_train7, pred_train7),
    ("Validation", Y_val7, pred_val7),
]:
    rows.append({
        "Dataset": name,
        "MAE (°C)": mean_absolute_error(actual, predicted),
        "RMSE (°C)": np.sqrt(mean_squared_error(actual, predicted)),
        "R²": r2_score(actual, predicted),
    })

print("Input: 7 giorni | Forecast: 30 giorni")
display(pd.DataFrame(rows).set_index("Dataset").round(3))

# MAE ai diversi orizzonti
rows = []

for h in [1, 7, 15, 30]:
    idx = h - 1
    rows.append({
        "Horizon (days)": h,
        "Train MAE (°C)": mean_absolute_error(
            Y_train7[:, idx], pred_train7[:, idx]
        ),
        "Validation MAE (°C)": mean_absolute_error(
            Y_val7[:, idx], pred_val7[:, idx]
        ),
    })

display(pd.DataFrame(rows).round(3))


Input: 7 giorni | Forecast: 30 giorni


,MAE (°C),RMSE (°C),R²
Dataset,,,
Training,2.143,2.690,0.877
Validation,2.161,2.736,0.875


,Horizon (days),Train MAE (°C),Validation MAE (°C)
0,1,1.097,1.126
1,7,2.181,2.164
2,15,2.225,2.259
3,30,2.231,2.285


In [19]:
import numpy as np
import pandas as pd

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor

HORIZON = 30
INPUT_DAYS_LIST = [1, 7, 14]

def build_windows(input_days):
    X_raw = data[features].to_numpy(dtype=float)
    y_raw = data[target].to_numpy(dtype=float)
    dates = data.index

    X_windows, Y_windows, start_dates = [], [], []

    for i in range(input_days, len(data) - HORIZON + 1):
        X_window = X_raw[i - input_days:i]
        Y_window = y_raw[i:i + HORIZON]

        if not np.isfinite(X_window).all():
            continue
        if not np.isfinite(Y_window).all():
            continue

        X_windows.append(X_window.reshape(-1))
        Y_windows.append(Y_window)
        start_dates.append(dates[i])

    X = np.asarray(X_windows, dtype=np.float32)
    Y = np.asarray(Y_windows, dtype=np.float32)
    start_dates = pd.DatetimeIndex(start_dates)

    end_dates = start_dates + pd.Timedelta(days=HORIZON - 1)

    train_mask = (
        (start_dates.year <= 2017)
        & (end_dates.year <= 2017)
    )

    val_mask = (
        (start_dates.year >= 2018)
        & (start_dates.year <= 2021)
        & (end_dates.year <= 2021)
    )

    return (
        X[train_mask], Y[train_mask],
        X[val_mask], Y[val_mask]
    )

In [20]:
models = {
    "Linear Regression": LinearRegression(),

    "Ridge": make_pipeline(
        StandardScaler(),
        Ridge(alpha=10.0)
    ),

    "Random Forest": RandomForestRegressor(
        n_estimators=200,
        max_depth=15,
        min_samples_leaf=3,
        random_state=42,
        n_jobs=-1
    )
}

results = []
horizon_results = []
trained_models = {}

for input_days in INPUT_DAYS_LIST:
    X_train, Y_train, X_val, Y_val = build_windows(input_days)

    print(f"\nInput: {input_days} giorni")
    print(f"Train: {X_train.shape}, target: {Y_train.shape}")
    print(f"Validation: {X_val.shape}, target: {Y_val.shape}")

    for model_name, base_model in models.items():
        # Ogni esperimento riceve un modello nuovo
        import sklearn.base
        model = sklearn.base.clone(base_model)

        model.fit(X_train, Y_train)

        pred_train = model.predict(X_train)
        pred_val = model.predict(X_val)

        results.append({
            "Model": model_name,
            "Input days": input_days,
            "Train MAE": mean_absolute_error(Y_train, pred_train),
            "Validation MAE": mean_absolute_error(Y_val, pred_val),
            "Validation RMSE": np.sqrt(
                mean_squared_error(Y_val, pred_val)
            ),
            "Validation R2": r2_score(Y_val, pred_val)
        })

        # Errori ai singoli orizzonti
        for h in [1, 7, 15, 30]:
            horizon_results.append({
                "Model": model_name,
                "Input days": input_days,
                "Horizon": h,
                "Train MAE": mean_absolute_error(
                    Y_train[:, h - 1],
                    pred_train[:, h - 1]
                ),
                "Validation MAE": mean_absolute_error(
                    Y_val[:, h - 1],
                    pred_val[:, h - 1]
                )
            })

        trained_models[(model_name, input_days)] = model

results_df = pd.DataFrame(results).sort_values(
    "Validation MAE"
).reset_index(drop=True)

horizon_df = pd.DataFrame(horizon_results)

display(results_df.round(4))


Input: 1 giorni
Train: (10168, 22), target: (10168, 30)
Validation: (1432, 22), target: (1432, 30)

Input: 7 giorni
Train: (10162, 154), target: (10162, 30)
Validation: (1432, 154), target: (1432, 30)

Input: 14 giorni
Train: (10155, 308), target: (10155, 30)
Validation: (1432, 308), target: (1432, 30)


,Model,Input days,Train MAE,Validation MAE,Validation RMSE,Validation R2
0,Linear Regression,1,2.1429,2.1606,2.7362,0.8749
1,Ridge,1,2.1442,2.1612,2.7378,0.8747
2,Ridge,7,2.1266,2.1661,2.7361,0.8749
3,Linear Regression,7,2.1153,2.1701,2.7337,0.8751
4,Ridge,14,2.1097,2.1707,2.7345,0.8750
5,Linear Regression,14,2.0962,2.1726,2.7337,0.8751
6,Random Forest,1,1.1566,2.2818,2.8971,0.8597
7,Random Forest,7,1.1475,2.2957,2.9152,0.8580
8,Random Forest,14,1.1415,2.3075,2.9363,0.8559


In [21]:
display(
    horizon_df
    .sort_values(["Horizon", "Validation MAE"])
    .round(4)
)

,Model,Input days,Horizon,Train MAE,Validation MAE
12,Linear Regression,7,1,1.0485,1.0853
24,Linear Regression,14,1,1.0426,1.0872
16,Ridge,7,1,1.0896,1.1241
0,Linear Regression,1,1,1.0968,1.1264
28,Ridge,14,1,1.0868,1.1264
4,Ridge,1,1,1.1034,1.1352
8,Random Forest,1,1,0.9839,1.4829
20,Random Forest,7,1,1.0015,1.6002
32,Random Forest,14,1,1.0112,1.6645
5,Ridge,1,7,2.1814,2.1637


In [22]:
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.multioutput import MultiOutputRegressor

hgb_results = []

for input_days in INPUT_DAYS_LIST:
    X_train, Y_train, X_val, Y_val = build_windows(input_days)

    model = MultiOutputRegressor(
        HistGradientBoostingRegressor(
            max_iter=100,
            learning_rate=0.05,
            max_leaf_nodes=15,
            l2_regularization=1.0,
            random_state=42
        ),
        n_jobs=-1
    )

    model.fit(X_train, Y_train)
    pred_val = model.predict(X_val)

    hgb_results.append({
        "Model": "HistGradientBoosting",
        "Input days": input_days,
        "Validation MAE": mean_absolute_error(Y_val, pred_val),
        "Validation RMSE": np.sqrt(
            mean_squared_error(Y_val, pred_val)
        ),
        "Validation R2": r2_score(Y_val, pred_val)
    })

hgb_df = pd.DataFrame(hgb_results)

display(hgb_df.round(4))

,Model,Input days,Validation MAE,Validation RMSE,Validation R2
0,HistGradientBoosting,1,2.1645,2.7398,0.8745
1,HistGradientBoosting,7,2.1603,2.7326,0.8752
2,HistGradientBoosting,14,2.1630,2.7392,0.8746


In [23]:
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_absolute_error

hgb_horizon_results = []

for input_days in [1, 7, 14]:
    X_train, Y_train, X_val, Y_val = build_windows(input_days)

    model = MultiOutputRegressor(
        HistGradientBoostingRegressor(
            max_iter=100,
            learning_rate=0.05,
            max_leaf_nodes=15,
            l2_regularization=1.0,
            random_state=42
        ),
        n_jobs=-1
    )

    model.fit(X_train, Y_train)
    pred_val = model.predict(X_val)

    for h in [1, 7, 15, 30]:
        hgb_horizon_results.append({
            "Model": "HistGradientBoosting",
            "Input days": input_days,
            "Horizon": h,
            "Validation MAE": mean_absolute_error(
                Y_val[:, h - 1],
                pred_val[:, h - 1]
            )
        })

hgb_horizon_df = pd.DataFrame(hgb_horizon_results)
display(hgb_horizon_df.round(4))

,Model,Input days,Horizon,Validation MAE
0,HistGradientBoosting,1,1,1.1501
1,HistGradientBoosting,1,7,2.1699
2,HistGradientBoosting,1,15,2.2343
3,HistGradientBoosting,1,30,2.2931
4,HistGradientBoosting,7,1,1.1420
5,HistGradientBoosting,7,7,2.1517
6,HistGradientBoosting,7,15,2.2504
7,HistGradientBoosting,7,30,2.2638
8,HistGradientBoosting,14,1,1.1363
9,HistGradientBoosting,14,7,2.1682


In [24]:
import numpy as np
import pandas as pd

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor
from sklearn.compose import TransformedTargetRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
from sklearn.exceptions import ConvergenceWarning
import warnings

INPUT_DAYS_LIST = [1, 7, 14]
HORIZON = 30

mlp_results = []
mlp_horizon_results = []
mlp_models = {}

for input_days in INPUT_DAYS_LIST:

    X_train, Y_train, X_val, Y_val = build_windows(input_days)

    # Standardizzazione degli input + rete neurale
    mlp_pipeline = make_pipeline(
        StandardScaler(),
        MLPRegressor(
            hidden_layer_sizes=(128, 64),
            activation="relu",
            solver="adam",
            alpha=0.001,
            learning_rate_init=0.001,
            max_iter=300,
            early_stopping=False,
            random_state=42
        )
    )

    # Standardizziamo anche i 30 target durante il training
    model = TransformedTargetRegressor(
        regressor=mlp_pipeline,
        transformer=StandardScaler()
    )

    print(f"\nTraining MLP con input di {input_days} giorni...")

    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always", ConvergenceWarning)
        model.fit(X_train, Y_train)

    convergence_warning = any(
        issubclass(w.category, ConvergenceWarning)
        for w in caught
    )

    pred_train = model.predict(X_train)
    pred_val = model.predict(X_val)

    mlp_models[input_days] = model

    mlp_results.append({
        "Model": "MLP (128, 64)",
        "Input days": input_days,
        "Train MAE": mean_absolute_error(Y_train, pred_train),
        "Validation MAE": mean_absolute_error(Y_val, pred_val),
        "Validation RMSE": np.sqrt(
            mean_squared_error(Y_val, pred_val)
        ),
        "Validation R2": r2_score(Y_val, pred_val),
        "Iterations": model.regressor_.steps[-1][1].n_iter_,
        "Convergence warning": convergence_warning
    })

    for h in [1, 7, 15, 30]:
        mlp_horizon_results.append({
            "Input days": input_days,
            "Horizon": h,
            "Train MAE": mean_absolute_error(
                Y_train[:, h - 1],
                pred_train[:, h - 1]
            ),
            "Validation MAE": mean_absolute_error(
                Y_val[:, h - 1],
                pred_val[:, h - 1]
            )
        })

mlp_results_df = (
    pd.DataFrame(mlp_results)
    .sort_values("Validation MAE")
    .reset_index(drop=True)
)

mlp_horizon_df = pd.DataFrame(mlp_horizon_results)

display(mlp_results_df.round(4))
display(mlp_horizon_df.round(4))


Training MLP con input di 1 giorni...

Training MLP con input di 7 giorni...

Training MLP con input di 14 giorni...


,Model,Input days,Train MAE,Validation MAE,Validation RMSE,Validation R2,Iterations,Convergence warning
0,"MLP (128, 64)",1,1.7905,2.4280,3.0832,0.8411,207,False
1,"MLP (128, 64)",14,1.5812,2.5186,3.1587,0.8332,163,False
2,"MLP (128, 64)",7,1.6021,2.6019,3.3168,0.8161,252,False


,Input days,Horizon,Train MAE,Validation MAE
0,1,1,1.0752,1.1550
1,1,7,1.8102,2.4683
2,1,15,1.8326,2.5999
3,1,30,1.9376,2.4599
4,7,1,1.1350,1.3750
5,7,7,1.6012,2.5518
6,7,15,1.6378,2.7418
7,7,30,1.7949,2.6144
8,14,1,1.1446,1.3742
9,14,7,1.5716,2.6171


In [25]:
all_results_df = pd.concat(
    [
        results_df[
            [
                "Model", "Input days", "Train MAE",
                "Validation MAE", "Validation RMSE",
                "Validation R2"
            ]
        ],
        hgb_df[
            [
                "Model", "Input days", "Validation MAE",
                "Validation RMSE", "Validation R2"
            ]
        ].assign(**{"Train MAE": np.nan})[
            [
                "Model", "Input days", "Train MAE",
                "Validation MAE", "Validation RMSE",
                "Validation R2"
            ]
        ],
        mlp_results_df[
            [
                "Model", "Input days", "Train MAE",
                "Validation MAE", "Validation RMSE",
                "Validation R2"
            ]
        ]
    ],
    ignore_index=True
).sort_values("Validation MAE").reset_index(drop=True)

display(all_results_df.round(4))

,Model,Input days,Train MAE,Validation MAE,Validation RMSE,Validation R2
0,HistGradientBoosting,7,NaN,2.1603,2.7326,0.8752
1,Linear Regression,1,2.1429,2.1606,2.7362,0.8749
2,Ridge,1,2.1442,2.1612,2.7378,0.8747
3,HistGradientBoosting,14,NaN,2.1630,2.7392,0.8746
4,HistGradientBoosting,1,NaN,2.1645,2.7398,0.8745
5,Ridge,7,2.1266,2.1661,2.7361,0.8749
6,Linear Regression,7,2.1153,2.1701,2.7337,0.8751
7,Ridge,14,2.1097,2.1707,2.7345,0.8750
8,Linear Regression,14,2.0962,2.1726,2.7337,0.8751
9,Random Forest,1,1.1566,2.2818,2.8971,0.8597


In [26]:

import numpy as np
import pandas as pd

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from sklearn.multioutput import MultiOutputRegressor
from sklearn.compose import TransformedTargetRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Prima prova: un solo giorno di input
input_days = 1
X_train, Y_train, X_val, Y_val = build_windows(input_days)

# Sottoinsieme deterministico per contenere i tempi di calcolo
MAX_SAMPLES = 3000
if len(X_train) > MAX_SAMPLES:
    indices = np.linspace(
        0, len(X_train) - 1, MAX_SAMPLES, dtype=int
    )
    X_train_svr = X_train[indices]
    Y_train_svr = Y_train[indices]
else:
    X_train_svr, Y_train_svr = X_train, Y_train

kernels = {
    "SVR linear": SVR(kernel="linear", C=1.0, epsilon=0.1),
    "SVR RBF": SVR(kernel="rbf", C=1.0, epsilon=0.1, gamma="scale"),
    "SVR polynomial": SVR(
        kernel="poly", degree=2, C=1.0,
        epsilon=0.1, gamma="scale"
    ),
}

svr_results = []
svr_models = {}

for name, svr in kernels.items():
    model = TransformedTargetRegressor(
        regressor=make_pipeline(
            StandardScaler(),
            MultiOutputRegressor(svr, n_jobs=-1)
        ),
        transformer=StandardScaler()
    )

    print(f"Training {name}...")
    model.fit(X_train_svr, Y_train_svr)
    pred_val = model.predict(X_val)

    svr_models[name] = model
    svr_results.append({
        "Model": name,
        "Input days": input_days,
        "Training samples": len(X_train_svr),
        "Validation MAE": mean_absolute_error(Y_val, pred_val),
        "Validation RMSE": np.sqrt(mean_squared_error(Y_val, pred_val)),
        "Validation R2": r2_score(Y_val, pred_val)
    })

svr_results_df = (
    pd.DataFrame(svr_results)
    .sort_values("Validation MAE")
    .reset_index(drop=True)
)

display(svr_results_df.round(4))


Training SVR linear...
Training SVR RBF...
Training SVR polynomial...


,Model,Input days,Training samples,Validation MAE,Validation RMSE,Validation R2
0,SVR RBF,1,3000,2.1419,2.7159,0.8767
1,SVR linear,1,3000,2.1680,2.7462,0.8740
2,SVR polynomial,1,3000,5.0239,6.2881,0.3391


In [30]:

import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# Reproducibility
SEED = 42
tf.keras.utils.set_random_seed(SEED)

INPUT_DAYS_LIST = [7, 14]
HORIZON = 30
N_FEATURES = len(features)

rnn_results = []
rnn_horizon_results = []
rnn_models = {}

for input_days in INPUT_DAYS_LIST:
    X_train, Y_train, X_val, Y_val = build_windows(input_days)

    # (samples, input_days, n_features)
    X_train_seq = X_train.reshape(-1, input_days, N_FEATURES)
    X_val_seq = X_val.reshape(-1, input_days, N_FEATURES)

    # Standardize features using training data only
    x_scaler = StandardScaler()
    x_scaler.fit(X_train_seq.reshape(-1, N_FEATURES))

    X_train_scaled = x_scaler.transform(
        X_train_seq.reshape(-1, N_FEATURES)
    ).reshape(X_train_seq.shape)

    X_val_scaled = x_scaler.transform(
        X_val_seq.reshape(-1, N_FEATURES)
    ).reshape(X_val_seq.shape)

    # Standardize target temperatures
    y_scaler = StandardScaler()
    y_scaler.fit(Y_train.reshape(-1, 1))

    Y_train_scaled = y_scaler.transform(
        Y_train.reshape(-1, 1)
    ).reshape(Y_train.shape)

    Y_val_scaled = y_scaler.transform(
        Y_val.reshape(-1, 1)
    ).reshape(Y_val.shape)

    # Define the model
    model = tf.keras.Sequential([
        tf.keras.layers.Input(
            shape=(input_days, N_FEATURES)
        ),
        tf.keras.layers.SimpleRNN(
            32,
            activation="tanh"
        ),
        tf.keras.layers.Dense(HORIZON)
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss="mse"
    )

    early_stopping = tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=7,
        restore_best_weights=True
    )

    print(f"\nTraining SimpleRNN with {input_days} input days...")

    history = model.fit(
        X_train_scaled,
        Y_train_scaled,
        validation_data=(X_val_scaled, Y_val_scaled),
        epochs=50,
        batch_size=128,
        callbacks=[early_stopping],
        shuffle=True,
        verbose=0
    )

    # Predictions in standardized units
    pred_train_scaled = model.predict(
        X_train_scaled, verbose=0
    )
    pred_val_scaled = model.predict(
        X_val_scaled, verbose=0
    )

    # Convert predictions back to Celsius
    pred_train = y_scaler.inverse_transform(
        pred_train_scaled.reshape(-1, 1)
    ).reshape(Y_train.shape)

    pred_val = y_scaler.inverse_transform(
        pred_val_scaled.reshape(-1, 1)
    ).reshape(Y_val.shape)

    rnn_models[input_days] = {
        "model": model,
        "x_scaler": x_scaler,
        "y_scaler": y_scaler
    }

    rnn_results.append({
        "Model": "SimpleRNN",
        "Input days": input_days,
        "Train MAE": mean_absolute_error(Y_train, pred_train),
        "Validation MAE": mean_absolute_error(Y_val, pred_val),
        "Validation RMSE": np.sqrt(
            mean_squared_error(Y_val, pred_val)
        ),
        "Validation R2": r2_score(Y_val, pred_val),
        "Epochs": len(history.history["loss"])
    })

    for h in [1, 7, 15, 30]:
        rnn_horizon_results.append({
            "Input days": input_days,
            "Horizon": h,
            "Train MAE": mean_absolute_error(
                Y_train[:, h - 1], pred_train[:, h - 1]
            ),
            "Validation MAE": mean_absolute_error(
                Y_val[:, h - 1], pred_val[:, h - 1]
            )
        })

rnn_results_df = (
    pd.DataFrame(rnn_results)
    .sort_values("Validation MAE")
    .reset_index(drop=True)
)

rnn_horizon_df = pd.DataFrame(rnn_horizon_results)

display(rnn_results_df.round(4))
display(rnn_horizon_df.round(4))



Training SimpleRNN with 7 input days...

Training SimpleRNN with 14 input days...


,Model,Input days,Train MAE,Validation MAE,Validation RMSE,Validation R2,Epochs
0,SimpleRNN,14,2.1026,2.1190,2.6798,0.8800,41
1,SimpleRNN,7,2.0944,2.1304,2.6916,0.8789,42


,Input days,Horizon,Train MAE,Validation MAE
0,7,1,1.1771,1.2091
1,7,7,2.1310,2.1421
2,7,15,2.1511,2.2041
3,7,30,2.1824,2.2170
4,14,1,1.1721,1.1892
5,14,7,2.1566,2.1474
6,14,15,2.1792,2.1663
7,14,30,2.1916,2.2182


In [31]:
comparison_df = pd.concat(
    [
        results_df[
            ["Model", "Input days", "Validation MAE",
             "Validation RMSE", "Validation R2"]
        ],
        hgb_df[
            ["Model", "Input days", "Validation MAE",
             "Validation RMSE", "Validation R2"]
        ],
        mlp_results_df[
            ["Model", "Input days", "Validation MAE",
             "Validation RMSE", "Validation R2"]
        ],
        svr_results_df[
            ["Model", "Input days", "Validation MAE",
             "Validation RMSE", "Validation R2"]
        ],
        rnn_results_df[
            ["Model", "Input days", "Validation MAE",
             "Validation RMSE", "Validation R2"]
        ]
    ],
    ignore_index=True
).sort_values("Validation MAE").reset_index(drop=True)

display(comparison_df.round(4))


,Model,Input days,Validation MAE,Validation RMSE,Validation R2
0,SimpleRNN,14,2.1190,2.6798,0.8800
1,SimpleRNN,7,2.1304,2.6916,0.8789
2,SVR RBF,1,2.1419,2.7159,0.8767
3,HistGradientBoosting,7,2.1603,2.7326,0.8752
4,Linear Regression,1,2.1606,2.7362,0.8749
5,Ridge,1,2.1612,2.7378,0.8747
6,HistGradientBoosting,14,2.1630,2.7392,0.8746
7,HistGradientBoosting,1,2.1645,2.7398,0.8745
8,Ridge,7,2.1661,2.7361,0.8749
9,SVR linear,1,2.1680,2.7462,0.8740


In [32]:

import numpy as np
import pandas as pd

from xgboost import XGBRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

INPUT_DAYS_LIST = [1, 7, 14]
HORIZON = 30

xgb_results = []
xgb_horizon_results = []
xgb_models = {}

for input_days in INPUT_DAYS_LIST:
    X_train, Y_train, X_val, Y_val = build_windows(input_days)

    base_model = XGBRegressor(
        objective="reg:squarederror",
        n_estimators=250,
        max_depth=3,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=5.0,
        random_state=42,
        n_jobs=-1,
        tree_method="hist"
    )

    model = MultiOutputRegressor(
        base_model,
        n_jobs=1
    )

    print(f"\nTraining XGBoost con {input_days} giorni di input...")
    model.fit(X_train, Y_train)

    pred_train = model.predict(X_train)
    pred_val = model.predict(X_val)

    xgb_models[input_days] = model

    xgb_results.append({
        "Model": "XGBoost",
        "Input days": input_days,
        "Train MAE": mean_absolute_error(Y_train, pred_train),
        "Validation MAE": mean_absolute_error(Y_val, pred_val),
        "Validation RMSE": np.sqrt(
            mean_squared_error(Y_val, pred_val)
        ),
        "Validation R2": r2_score(Y_val, pred_val)
    })

    for h in [1, 7, 15, 30]:
        xgb_horizon_results.append({
            "Input days": input_days,
            "Horizon": h,
            "Train MAE": mean_absolute_error(
                Y_train[:, h - 1],
                pred_train[:, h - 1]
            ),
            "Validation MAE": mean_absolute_error(
                Y_val[:, h - 1],
                pred_val[:, h - 1]
            )
        })

xgb_results_df = (
    pd.DataFrame(xgb_results)
    .sort_values("Validation MAE")
    .reset_index(drop=True)
)

xgb_horizon_df = pd.DataFrame(xgb_horizon_results)

display(xgb_results_df.round(4))
display(xgb_horizon_df.round(4))



Training XGBoost con 1 giorni di input...

Training XGBoost con 7 giorni di input...

Training XGBoost con 14 giorni di input...


,Model,Input days,Train MAE,Validation MAE,Validation RMSE,Validation R2
0,XGBoost,1,1.9050,2.1502,2.7202,0.8763
1,XGBoost,7,1.8676,2.1557,2.7220,0.8762
2,XGBoost,14,1.8480,2.1668,2.7346,0.8750


,Input days,Horizon,Train MAE,Validation MAE
0,1,1,1.0423,1.1311
1,1,7,1.9603,2.1410
2,1,15,1.9517,2.1996
3,1,30,1.9688,2.2894
4,7,1,1.0288,1.1304
5,7,7,1.9047,2.1438
6,7,15,1.9138,2.2443
7,7,30,1.9274,2.2712
8,14,1,1.0248,1.1390
9,14,7,1.8765,2.1632


In [33]:
comparison_df = pd.concat(
    [
        comparison_df,
        xgb_results_df[
            ["Model", "Input days", "Validation MAE",
             "Validation RMSE", "Validation R2"]
        ]
    ],
    ignore_index=True
).sort_values("Validation MAE").reset_index(drop=True)

display(comparison_df.round(4))


,Model,Input days,Validation MAE,Validation RMSE,Validation R2
0,SimpleRNN,14,2.1190,2.6798,0.8800
1,SimpleRNN,7,2.1304,2.6916,0.8789
2,SVR RBF,1,2.1419,2.7159,0.8767
3,XGBoost,1,2.1502,2.7202,0.8763
4,XGBoost,7,2.1557,2.7220,0.8762
5,HistGradientBoosting,7,2.1603,2.7326,0.8752
6,Linear Regression,1,2.1606,2.7362,0.8749
7,Ridge,1,2.1612,2.7378,0.8747
8,HistGradientBoosting,14,2.1630,2.7392,0.8746
9,HistGradientBoosting,1,2.1645,2.7398,0.8745
